In [ ]:
import numpy as np
from scipy.special import roots_hermitenorm, eval_hermitenorm, factorial
from scipy.optimize import brentq

def gaussian_expectation(f, n_quad=200):
    """
    Compute E[f(Z)] for Z ~ N(0,1)
    """
    x, w = roots_hermitenorm(n_quad)
    return np.sum(w * f(x)) / np.sqrt(2 * np.pi)

def normalize_transform(f, n_quad=200):
    """
    Return a centered variance-1 version of f
    """

    mean = gaussian_expectation(f, n_quad)
    var = gaussian_expectation(
        lambda z: (f(z)-mean)**2,
        n_quad
    )
    sd = np.sqrt(var)

    return lambda z: (f(z)-mean)/sd

def get_hermite_coeff(f, k, n_quad=200):
    """
    Orthonormal Hermite coefficient

    c_k = E[f(Z)He_k(Z)] / sqrt(k!)
    where Z ~ N(0, 1)
    """
    x, w = roots_hermitenorm(n_quad)

    expectation = (
        np.sum(w * f(x) * eval_hermitenorm(k, x))
        / np.sqrt(2 * np.pi)
    )

    return expectation / np.sqrt(factorial(k, exact=False))

def hermite_distribution(f, max_order=20, n_quad=150):
    """
    Calculate Hermite coefficients and energies for a
    centered, variance-normalized transform.

    Returns
    dict containing:
        coefficients
        energies
        nonlinear_energy
        first_order_energy
        captured_energy
    """

    g = normalize_transform(f, n_quad)

    coeffs = np.array([
        get_hermite_coeff(g, k, n_quad)
        for k in range(max_order + 1)
    ])

    energies = coeffs**2

    return {
        "coefficients": coeffs,
        "energies": energies,
        "first_order_energy": energies[1],
        "nonlinear_energy": 1 - energies[1],
        "captured_energy": np.sum(energies)
    }

def find_strength(
    transform,
    target_nonlinear_energy,
    strength_bounds,
    max_order=20,
    n_quad=150
):
    """
    Find theta such that

        1 - rho^2 = target_nonlinear_energy

    where rho = Corr(Z, T_theta(Z)).

    Parameters
    ----------
    transform:
        Function of form transform(z, strength)

    target_nonlinear_energy:
        Desired value of 1 - rho^2

    strength_bounds:
        Tuple (low, high) in which to search

    max_order:
        Maximum Hermite order returned

    Returns
    -------
    result : dict
        strength
        nonlinear_energy
        first_order_energy
        coefficients
        energies
        captured_energy
    """

    def objective(strength):

        f = lambda z: transform(z, strength)

        result = hermite_distribution(
            f,
            max_order=max_order,
            n_quad=n_quad
        )

        return (
            result["nonlinear_energy"]
            - target_nonlinear_energy
        )

    strength = brentq(
        objective,
        strength_bounds[0],
        strength_bounds[1]
    )

    f = lambda z: transform(z, strength)

    result = hermite_distribution(
        f,
        max_order=max_order,
        n_quad=n_quad
    )

    result["strength"] = strength

    return result

def cubic(z, strength):
    return z + strength * z**3


def quintic(z, strength):
    return z + strength * z**5


def exponential(z, strength):
    if strength == 0:
        return z
    return np.expm1(strength * z) / strength


def softplus(z, strength):
    if strength == 0:
        return z
    return np.logaddexp(0, strength * z) / strength


for transform in [cubic, quintic, exponential, softplus]:     
    print(transform)  
    for target in [0.05, 0.1, 0.15, 0.2, 0.3]:
            
        result = find_strength(
            transform=exponential,
            target_nonlinear_energy=target,
            strength_bounds=(0.0, 2.0),
            max_order=15
        )

        print("Required strength:", result["strength"])


<function cubic at 0x0000020A5B4FC9A0>
Required strength: 0.3189425386491139
Order  0: 0.00000000
Order  1: 0.95000000
Order  2: 0.04831906
Order  3: 0.00163841
Order  4: 0.00004167
Order  5: 0.00000085
Order  6: 0.00000001
Order  7: 0.00000000
Order  8: 0.00000000
Order  9: 0.00000000
Order 10: 0.00000000
Order 11: 0.00000000
Order 12: 0.00000000
Order 13: 0.00000000
Order 14: 0.00000000
Order 15: 0.00000000
Required strength: 0.45513350013401593
Order  0: 0.00000000
Order  1: 0.90000000
Order  2: 0.09321593
Order  3: 0.00643645
Order  4: 0.00033332
Order  5: 0.00001381
Order  6: 0.00000048
Order  7: 0.00000001
Order  8: 0.00000000
Order  9: 0.00000000
Order 10: 0.00000000
Order 11: 0.00000000
Order 12: 0.00000000
Order 13: 0.00000000
Order 14: 0.00000000
Order 15: 0.00000000
Required strength: 0.5627497349486706
Order  0: 0.00000000
Order  1: 0.85000000
Order  2: 0.13459209
Order  3: 0.01420787
Order  4: 0.00112486
Order  5: 0.00007125
Order  6: 0.00000376
Order  7: 0.00000017
Order 